# 02 · Carga y Verificación de Datos Crudos
**Tesis:** Medición del ciclo financiero en Perú
**Autor:** Roberto Samaniego | **Asesor:** Dr. Sergio Camiz

---

**Responsabilidad única de este notebook:** cargar `data/raw/series_bcrp_raw.csv`
y verificar su integridad (cobertura, nulos, duplicados, consistencia del índice).

Este notebook **no modifica ningún valor** del dataset crudo. No hay `dropna()`,
`resample()`, ni transformaciones de ningún tipo sobre `df_raw`. El único archivo
que se guarda es un *reporte* de verificación (metadatos sobre la calidad del
dato), nunca una versión alterada de los datos mismos.

## 1. Librerías y carga

In [1]:
import os
os.chdir('..')
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df_raw = pd.read_csv('data/raw/series_bcrp_raw.csv', index_col=0, parse_dates=True)
print(f'Dataset crudo cargado: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas')
df_raw.head()

Dataset crudo cargado: 228 filas x 18 columnas


,Crédito SF sector privado,Crédito empresarial,Crédito consumo,Crédito hipotecario,Tasa activa TAMN,Tasa pasiva TIPMN,Tasa referencia BCRP,Tipo de cambio,Índice BVL,IPC Lima,PBI desestacionalizado,Demanda interna,Reservas internacionales,Liquidez M1,Liquidez M2,Liquidez M3,Tasa interbancaria,IPC Subyacente
fecha,,,,,,,,,,,,,,,,,,
2007-01-01,69819.470429,39105.687117,11831.023259,7881.540785,23.750000,3.17,4.5,3.192500,13633.78,62.563161,96.170448,93.230311,17849.371719,17720.111612,82579.596567,124721.387082,4.47,62.675629
2007-02-01,70717.513741,39559.563157,12056.909321,7995.384322,23.570000,3.24,4.5,3.190300,15150.74,62.725492,96.288126,88.898969,18135.788339,18071.896172,85842.029725,127949.026374,4.48,62.837262
2007-03-01,72596.321022,40630.071977,12336.710861,8132.870391,23.400000,3.22,4.5,3.185614,17152.82,62.944301,97.831482,95.854561,18427.065849,19144.793538,89837.639732,132484.042161,4.51,63.047336
2007-04-01,74195.323389,41664.440265,12608.628826,8279.940038,22.781333,3.10,4.5,3.178289,20674.78,63.056472,97.441840,98.357011,19703.979779,19148.938407,94523.790386,138682.051720,4.51,63.127862
2007-05-01,75878.925497,42288.944366,13065.238278,8431.477703,22.130000,3.12,4.5,3.167523,20129.50,63.366629,100.527037,107.184473,21270.882177,18761.899696,96594.862494,142112.509184,4.49,63.182281


## 2. Cobertura temporal por serie

In [2]:
cobertura = pd.DataFrame({
    'Inicio':  df_raw.apply(lambda x: x.dropna().index.min()),
    'Fin':     df_raw.apply(lambda x: x.dropna().index.max()),
    'Obs':     df_raw.count(),
    'Nulos':   df_raw.isna().sum(),
    'Nulos %': (df_raw.isna().sum() / len(df_raw) * 100).round(1),
})
print('Cobertura y nulos por serie:')
cobertura

Cobertura y nulos por serie:


,Inicio,Fin,Obs,Nulos,Nulos %
Crédito SF sector privado,2007-01-01,2025-12-01,228,0,0.0
Crédito empresarial,2007-01-01,2025-12-01,228,0,0.0
Crédito consumo,2007-01-01,2025-12-01,228,0,0.0
Crédito hipotecario,2007-01-01,2025-12-01,228,0,0.0
Tasa activa TAMN,2007-01-01,2025-12-01,228,0,0.0
Tasa pasiva TIPMN,2007-01-01,2025-12-01,228,0,0.0
Tasa referencia BCRP,2007-01-01,2025-12-01,228,0,0.0
Tipo de cambio,2007-01-01,2025-12-01,228,0,0.0
Índice BVL,2007-01-01,2025-12-01,228,0,0.0
IPC Lima,2007-01-01,2025-12-01,228,0,0.0


## 3. Duplicados y consistencia del índice temporal

In [3]:
n_duplicados = df_raw.index.duplicated().sum()
es_monotono = df_raw.index.is_monotonic_increasing

print(f'Fechas duplicadas: {n_duplicados}')
print(f'Índice estrictamente creciente: {es_monotono}')

if n_duplicados > 0:
    print('\n⚠ Revisar fechas duplicadas antes de continuar al preprocesamiento:')
    print(df_raw[df_raw.index.duplicated(keep=False)].index.unique())

Fechas duplicadas: 0
Índice estrictamente creciente: True


## 4. Estadísticos descriptivos en niveles originales (sin transformar)

In [4]:
print('Estadísticos descriptivos — valores tal como los reporta el BCRP:')
df_raw.describe().round(2)

Estadísticos descriptivos — valores tal como los reporta el BCRP:


,Crédito SF sector privado,Crédito empresarial,Crédito consumo,Crédito hipotecario,Tasa activa TAMN,Tasa pasiva TIPMN,Tasa referencia BCRP,Tipo de cambio,Índice BVL,IPC Lima,PBI desestacionalizado,Demanda interna,Reservas internacionales,Liquidez M1,Liquidez M2,Liquidez M3,Tasa interbancaria,IPC Subyacente
count,228.00,228.00,228.00,228.00,228.00,228.00,228.00,228.00,228.00,228.00,228.00,228.00,228.00,228.00,228.00,228.00,228.00,228.00
mean,286574.55,161740.47,56730.94,39531.20,16.79,2.43,3.88,3.24,19168.74,86.87,149.93,161.84,59598.60,81907.82,308591.78,414114.57,3.89,87.26
std,133209.20,67053.85,28923.54,19645.03,3.34,0.72,1.80,0.40,5799.32,15.54,26.84,33.06,17068.56,44784.05,136046.55,179910.62,1.80,15.59
min,69819.47,39105.69,11831.02,7881.54,10.49,0.76,0.25,2.55,6671.72,62.56,96.17,88.90,17849.37,17720.11,82579.60,124721.39,0.11,62.68
25%,160988.10,104382.53,31205.51,21609.01,14.57,2.20,2.75,2.85,15385.47,73.74,127.59,136.06,48651.12,45025.01,179919.87,245582.34,2.77,73.46
50%,296132.70,165891.14,55629.20,40215.29,16.03,2.39,4.25,3.25,19322.03,85.87,154.59,167.83,63023.93,69395.02,300744.67,405290.48,4.22,87.26
75%,425403.59,237760.78,77820.68,55499.09,19.01,2.71,4.75,3.57,21430.47,95.26,172.82,187.06,72382.48,133127.10,437281.93,594966.19,4.70,97.00
max,479089.33,253176.10,107850.75,74632.28,24.33,4.08,7.75,4.11,43464.77,115.92,193.43,242.17,90898.18,178926.59,550978.61,737697.23,7.76,116.49


## 5. Reporte de verificación

Se guarda un reporte de calidad de dato — esto **no** altera `data/raw/`,
es un artefacto separado para trazabilidad y para mostrarle al asesor.

In [5]:
os.makedirs('reports/tables', exist_ok=True)
cobertura.to_csv('reports/tables/02_reporte_verificacion_bcrp.csv')
print('Reporte guardado en reports/tables/02_reporte_verificacion_bcrp.csv')
print()
print('IMPORTANTE: data/raw/series_bcrp_raw.csv permanece sin modificar.')
print('La limpieza y transformación real ocurre en 04_preprocesamiento.ipynb')

Reporte guardado en reports/tables/02_reporte_verificacion_bcrp.csv

IMPORTANTE: data/raw/series_bcrp_raw.csv permanece sin modificar.
La limpieza y transformación real ocurre en 04_preprocesamiento.ipynb


In [6]:
import pandas as pd

df = pd.read_csv('data/processed/dataset_estandarizado_trimestral.csv', index_col=0, parse_dates=True)
print(df.shape)
print(df.columns.tolist())
print(df.tail())

(72, 13)
['Crédito empresarial_log_diff', 'Crédito consumo_log_diff', 'Crédito hipotecario_log_diff', 'Tipo de cambio_log_diff', 'Índice BVL_log_diff', 'IPC Lima_log_diff', 'PBI desestacionalizado_log_diff', 'Demanda interna_log_diff', 'Reservas internacionales_log_diff', 'indice_precios_inmuebles_log_diff', 'Tasa activa TAMN_diff', 'Tasa pasiva TIPMN_diff', 'Tasa referencia BCRP_diff']
            Crédito empresarial_log_diff  Crédito consumo_log_diff  \
fecha                                                                
2024-12-31                     -1.098938                 -0.576463   
2025-03-31                     -0.982311                 -0.700307   
2025-06-30                     -0.051120                 -0.498457   
2025-09-30                     -0.071978                 -0.255131   
2025-12-31                     -0.329962                 -0.102779   

            Crédito hipotecario_log_diff  Tipo de cambio_log_diff  \
fecha                                             